In [5]:
lazy import pandas as pd
lazy import numpy as np
lazy import matplotlib.pyplot as plt
lazy import seaborn as sns

lazy from sklearn.model_selection import train_test_split, cross_val_score
lazy from sklearn.preprocessing import StandardScaler, OneHotEncoder
lazy from sklearn.linear_model import LinearRegression
lazy from sklearn.compose import ColumnTransformer
lazy from sklearn.pipeline import Pipeline
lazy from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All important library is imported succesfully!")



All important library is imported succesfully!


# Feature Selection Experiment

## Objective

The objective of this experiment is to identify the most useful features for predicting the Performance Index of students.

We will compare different feature combinations using Linear Regression to determine whether reducing the number of input features can maintain or improve predictive performance.

## Baseline Model

The baseline model uses all five input features:

1. Hours Studied
2. Previous Scores
3. Extracurricular Activities
4. Sleep Hours
5. Sample Question Papers Practiced

## Evaluation Metrics

We will evaluate each model using:

* Mean Absolute Error (MAE)
* Root Mean Squared Error (RMSE)
* R² Score
* Cross-validation R² Score

A good feature combination should achieve a high R² score and low MAE and RMSE while using fewer features where possible.

## Important Note

All experiments will use the same train-test split and preprocessing approach to ensure a fair comparison. Feature selection will be based on training data and cross-validation, with the test set reserved for final evaluation.


## We select 5 pairs of feature to experiment the model accuracy:-
1. All 5 features:- We already done this so we can do again in this notebook 
2. Three Features (P,H,Sleep)
3. Three Features (P,H,Sample)
4. Two Features
5. One Features



In [7]:
df = pd.read_csv('../data/processed/df.csv')
print("Dataset loaded successfulyy!")

Dataset loaded successfulyy!


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9873 entries, 0 to 9872
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     9873 non-null   int64  
 1   Previous Scores                   9873 non-null   int64  
 2   Extracurricular Activities        9873 non-null   str    
 3   Sleep Hours                       9873 non-null   int64  
 4   Sample Question Papers Practiced  9873 non-null   int64  
 5   Performance Index                 9873 non-null   float64
dtypes: float64(1), int64(4), str(1)
memory usage: 462.9 KB


## Baseline Model — All Five Features

The baseline Linear Regression model uses all five original input features.

| Metric                   | Baseline Result |
| ------------------------ | --------------: |
| MAE                      |          1.6470 |
| RMSE                     |          2.0751 |
| Test R² Score            |         0.98843 |
| Mean Cross-Validation R² |        0.988695 |

These results will serve as the reference for comparing reduced feature combinations.

Note: The baseline and reduced models must use the same dataset version, train-test split, and cross-validation strategy for a fair comparison.


## Experiment A vs Experiment B: Feature Reduction

**Experiment A — Baseline Model:** Uses all five original input features.

**Experiment B — Reduced Model:** Uses only `Previous Scores` and `Hours Studied`.

### Hypothesis

Previous Scores and Hours Studied may provide sufficient predictive information to achieve performance close to the baseline model.

### Goal

Compare MAE, RMSE, R², and cross-validation R² to determine whether reducing the number of features significantly affects predictive performance.


In [9]:
# Define target
y = df["Performance Index"]

# Experiment A: All five original features
features_A = [
    "Hours Studied",
    "Previous Scores",
    "Extracurricular Activities",
    "Sleep Hours",
    "Sample Question Papers Practiced"
]

# Experiment B: Only two features
features_B = [
    "Previous Scores",
    "Hours Studied"
]

X_A = df[features_A].copy()
X_B = df[features_B].copy()

print("Experiment A shape:", X_A.shape)
print("Experiment B shape:", X_B.shape)

Experiment A shape: (9873, 5)
Experiment B shape: (9873, 2)


In [10]:
X_train_A, X_test_A, y_train, y_test = train_test_split(
    X_A, y,
    test_size=0.2,
    random_state=42
)

X_train_B, X_test_B, _, _ = train_test_split(
    X_B, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train_A))
print("Testing rows:", len(X_test_A))

print("Experiment A training features:", X_train_A.shape[1])
print("Experiment B training features:", X_train_B.shape[1])

Training rows: 7898
Testing rows: 1975
Experiment A training features: 5
Experiment B training features: 2


In [11]:
def evaluate_model(X_train, X_test, y_train, y_test):

    numerical_features = X_train.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    transformers = []

    if numerical_features:
        transformers.append(
            ("num", StandardScaler(), numerical_features)
        )

    if categorical_features:
        transformers.append(
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "Test R²": r2,
        "Mean CV R²": cv_scores.mean()
    }

In [12]:
results_A = evaluate_model(
    X_train_A, X_test_A, y_train, y_test
)

results_B = evaluate_model(
    X_train_B, X_test_B, y_train, y_test
)

comparison = pd.DataFrame(
    [results_A, results_B],
    index=["Experiment A: All 5 Features",
           "Experiment B: 2 Features"]
)

display(comparison.round(5))

,MAE,RMSE,Test R²,Mean CV R²
Experiment A: All 5 Features,1.64697,2.07507,0.98843,0.98872
Experiment B: 2 Features,1.87983,2.36088,0.98502,0.98598


## Experiment A vs Experiment B — Results and Interpretation

Experiment A uses all five input features, whereas Experiment B uses only Previous Scores and Hours Studied.

Experiment B achieved a test R² score of 0.98502, compared with 0.98843 for Experiment A. However, the reduced model produced higher MAE and RMSE values.

These results suggest that Previous Scores and Hours Studied alone provide strong predictive information, but the remaining features contribute additional information that improves prediction performance.

### Conclusion

The two-feature model offers a simpler alternative with slightly lower predictive performance. Further experiments with three-feature combinations are required before selecting the final feature set.


In [13]:
features_C = [
    "Previous Scores",
    "Hours Studied",
    "Sleep Hours"
]

X_C = df[features_C].copy()

X_train_C, X_test_C, _, _ = train_test_split(
    X_C,
    y,
    test_size=0.2,
    random_state=42
)

print("Experiment C shape:", X_C.shape)
print("Training rows:", len(X_train_C))
print("Testing rows:", len(X_test_C))

Experiment C shape: (9873, 3)
Training rows: 7898
Testing rows: 1975


In [14]:
results_C = evaluate_model(
    X_train_C,
    X_test_C,
    y_train,
    y_test
)

print("Experiment C Results:")

for metric, value in results_C.items():
    print(f"{metric}: {value:.5f}")

Experiment C Results:
MAE: 1.75700
RMSE: 2.19759
Test R²: 0.98702
Mean CV R²: 0.98769


## Experiment C: Three-Feature Model

Experiment C uses Previous Scores, Hours Studied, and Sleep Hours.

The model achieved a test R² score of 0.98702 and a mean cross-validation R² score of 0.98769.

Compared with Experiment B, adding Sleep Hours improved predictive performance by reducing MAE and RMSE and increasing the R² score.

However, Experiment A, which uses all five features, still achieved better results.

### Conclusion

Sleep Hours provides some additional predictive value in this feature combination. Further experiments are required to determine whether Sample Question Papers Practiced can produce a better balance between model simplicity and predictive performance.


In [15]:
features_D = [
    "Previous Scores",
    "Hours Studied",
    "Sample Question Papers Practiced"
]

X_D = df[features_D].copy()

X_train_D, X_test_D, _, _ = train_test_split(
    X_D,
    y,
    test_size=0.2,
    random_state=42
)

print("Experiment D shape:", X_D.shape)
print("Training rows:", len(X_train_D))
print("Testing rows:", len(X_test_D))

results_D = evaluate_model(
    X_train_D,
    X_test_D,
    y_train,
    y_test
)

print("Experiment D Results:")

for metric, value in results_D.items():
    print(f"{metric}: {value:.5f}")

Experiment D shape: (9873, 3)
Training rows: 7898
Testing rows: 1975
Experiment D Results:
MAE: 1.80532
RMSE: 2.27694
Test R²: 0.98607
Mean CV R²: 0.98679


## Experiment D: Three-Feature Model with Sample Question Papers Practiced

Experiment D uses Previous Scores, Hours Studied, and Sample Question Papers Practiced.

The model achieved a test R² score of 0.98607 and a mean cross-validation R² score of 0.98679.

Compared with Experiment C, Experiment D achieved a lower R² score and higher MAE and RMSE values.

### Conclusion

For these three-feature combinations, the model using Sleep Hours performed better than the model using Sample Question Papers Practiced.

The five-feature baseline continues to achieve the best predictive performance. Experiment E will evaluate whether Previous Scores alone can provide a useful simplified model.


In [16]:
features_E = [
    "Previous Scores"
]

X_E = df[features_E].copy()

X_train_E, X_test_E, _, _ = train_test_split(
    X_E,
    y,
    test_size=0.2,
    random_state=42
)

print("Experiment E shape:", X_E.shape)
print("Training rows:", len(X_train_E))
print("Testing rows:", len(X_test_E))


results_E = evaluate_model(
    X_train_E,
    X_test_E,
    y_train,
    y_test
)

print("Experiment E Results:")

for metric, value in results_E.items():
    print(f"{metric}: {value:.5f}")

Experiment E shape: (9873, 1)
Training rows: 7898
Testing rows: 1975
Experiment E Results:
MAE: 6.70871
RMSE: 7.81734
Test R²: 0.83580
Mean CV R²: 0.83770


# Final Conclusion: Feature Selection Experiment

Five feature combinations were evaluated using Linear Regression to determine whether reducing the number of input features could maintain predictive performance.

The five-feature baseline achieved the best overall results, with a test R² score of 0.98843, MAE of 1.64697, and RMSE of 2.07507.

The two-feature model, using Previous Scores and Hours Studied, achieved a test R² score of 0.98502. Although its performance was slightly lower, it demonstrated that a simpler model could still explain a large proportion of the target's variation.

The three-feature experiments achieved test R² scores of 0.98702 and 0.98607, respectively. The single-feature model using Previous Scores alone performed substantially worse, with a test R² score of 0.83580.

## Final Decision

Based on the current experimental results, the five-feature model is selected as the preferred candidate because it provides the best predictive performance among the tested feature combinations.

The two-feature model remains a possible alternative when reducing input requirements is more important than achieving the lowest prediction error.

These conclusions apply to the evaluated dataset. Performance on real students' academic records would require independent validation.
